# NOT training with smoothing

Baseline of Choi et al. (2025): GDmax on the Semi-Dual problem with Gaussian noise added to the source samples, annealed from $\sigma = 0.2$ to $0.05$.

**How to run.** Run the setup cells (imports, *Distributions*, *Architectures*), then the training section, which seeds the RNG with `SEED` and builds a fresh `T` and `V` (`f`).

In [ ]:
import os, sys
sys.path.append("..")

import numpy as np
import torch
import torch.nn as nn
import matplotlib.pyplot as plt
from IPython.display import clear_output

device = "cuda" if torch.cuda.is_available() else "cpu"

from src.utils import cost, show_mapping , grad_norm, cosine_lr, sigma_schedule, plot_3d_function
from src.cost import compute_transport_cost, compute_transport_cost_noise, sinkhorn_w2, compute_w2_sinkhorn, compute_w2_sinkhorn_noise
from src.datasets import toy_dataset
from src.train import train_gdmax


## Distributions

Pick the data with `DATASET`: `"perpendicular"`, `"horizontal"`, `"one_to_many"` or `"grid"` (Choi et al., 2025). `toy_dataset` returns the two samplers and the optimal transport cost $W_2^2(\mu, \nu)$, which `evaluate` uses for $D_\text{cost}$.

In [ ]:
DATASET = "perpendicular"   # "perpendicular", "horizontal", "one_to_many" or "grid"
SEED = 0
# Step size of both players per dataset (lowest mean D_target over 3 seeds among 1e-4, 3e-4, 1e-3).
LR = {
    "perpendicular": 1e-4,  # D_target 0.0366 over 3 seeds
    "horizontal": 1e-3,  # D_target 0.0239 over 3 seeds
    "one_to_many": 1e-4,  # D_target 0.0684 over 3 seeds
    "grid": 3e-4,  # D_target 0.0399 over 3 seeds
}

sample_mu, sample_nu, W2_SQ = toy_dataset(DATASET)


def evaluate(T, sigma=0.0, n_mc=10, n_samples=2048):
    """Transport cost E||x - T(x + sigma z)||^2, D_cost = |cost - W2^2| and D_target
    (Sinkhorn W2 between T#mu and nu), as mean +- std over n_mc Monte Carlo estimates.
    sigma = 0.05 evaluates the smoothed map at the final noise level."""
    costs = [compute_transport_cost(T, sample_mu, sigma=sigma, n_samples=n_samples) for _ in range(n_mc)]
    d_target = [compute_w2_sinkhorn(T, sample_mu, sample_nu, sigma=sigma, n_samples=n_samples)
                for _ in range(n_mc)]
    d_cost = [abs(c - W2_SQ) for c in costs]
    print(f"[input noise sigma = {sigma}]")
    print(f"transport cost: {np.mean(costs):.4f} ± {np.std(costs):.4f}  (W2^2 = {W2_SQ:.4f})")
    print(f"D_cost:         {np.mean(d_cost):.4f} ± {np.std(d_cost):.4f}")
    print(f"D_target:       {np.mean(d_target):.4f} ± {np.std(d_target):.4f}")


## Architectures

One-hidden-layer networks as in Choi et al. (2025); `T` is deterministic and the noise enters through its input.

In [ ]:
def get_hidden_dim(d):
    if d in [2, 4]:
        return 256
    else:
        return 1024


# Potential function v_phi: R^d -> R
class Critic(nn.Module):
    def __init__(self, d):
        super().__init__()
        h = get_hidden_dim(d)

        self.net = nn.Sequential(
            nn.Linear(d, h),
            nn.ReLU(),
            nn.Linear(h, 1)
        )

    def forward(self, x):
        return self.net(x)


# Transport map T_theta: R^d -> R^d
class Transport(nn.Module):
    def __init__(self, d):
        super().__init__()
        h = get_hidden_dim(d)

        self.net = nn.Sequential(
            nn.Linear(d, h),
            nn.ReLU(),
            nn.Linear(h, d)
        )

    def forward(self, x):
        return self.net(x)


d = 2


## GDmax Training

$K = 20$ Adam descent steps on `T` per Adam ascent step on `f`, each on a fresh batch with fresh noise (`train_gdmax` with `noise_fn`).

In [ ]:
torch.manual_seed(SEED)
T = Transport(d).to(device)
f = Critic(d).to(device)

sigma_state = {"value": 0.2}

def noise_fn(step, x):
    x = x + sigma_state["value"] * torch.randn_like(x)
    if (step + 1) % 500 == 0:
        sigma_state["value"] = sigma_schedule(step + 1, 20000)
    return x

history = train_gdmax(
    T, f, sample_mu, sample_nu, cost,
    n_steps=20000, lr_T=LR[DATASET], lr_f=LR[DATASET], K=20,
    batch_size_x=128, batch_size_y=128,
    noise_fn=noise_fn,
    callback=lambda step, T, f: show_mapping(T, sample_mu, sample_nu),
)

sigma = sigma_state["value"]

## Plot Results

In [ ]:
show_mapping(T,sample_mu,sample_nu, option = True)

In [ ]:
option = True

x = sample_mu(1000).to(device)
z = torch.randn(1000, d, device=device)

x_tilde = x + sigma * z

outputs = []
for _ in range(5):
    outputs.append(T(x_tilde).detach().cpu())

outputs = torch.cat(outputs, dim=0)
x = x.cpu()
y = sample_nu(1000).cpu()
x_tilde = x_tilde.cpu()

plt.figure()

if (option == True):
    for i in range(50):
        plt.arrow(x[i,0], x[i,1],
              outputs[i,0]-x[i,0],
              outputs[i,1]-x[i,1],
              color='gray', alpha=0.5, head_width=0, length_includes_head=True)

plt.scatter(x[:,0], x[:,1], label=r"$x \sim \mu$", s=1, alpha=0.5)
plt.scatter(x_tilde[:,0], x_tilde[:,1], label=r"$x + z$", s=1, alpha=0.5, color = 'skyblue')
plt.scatter(y[:,0], y[:,1], label=r"$y \sim \nu$", s=1, alpha=0.5)
plt.scatter(outputs[:,0], outputs[:,1], label="T(x+z)", s=1, alpha=0.5)
plt.axis('equal')
plt.legend()
plt.show()

In [ ]:
plot_3d_function(f)


In [ ]:
evaluate(T)              # the map on clean inputs
evaluate(T, sigma=0.05)  # the smoothed map T(x + 0.05 z)
